# EPA Station Monitoring + Climate Data Merge

**Goal:** Merge EPA water quality measurements with ISU climate data.

**Join keys:**
- **Location:** Each EPA monitoring location (lat/lon) is spatially matched to the nearest ISU climate station using a haversine nearest-neighbor search.
- **Time:** `ActivityStartDate` (EPA WQ) == `day` (climate).

**Input files:**
- `data/tabular/water-quality/raw/epa-wq.csv`
- `data/tabular/water-quality/raw/epa-stations.csv`
- `data/tabular/climate/raw/isu-climate.csv`

**Output:** `data/tabular/merged/epa-climate-merged.csv`

In [1]:
import numpy as np
import pandas as pd
import requests
import json
import os
from scipy.spatial import cKDTree

## Step 1: Load Merged EPA Water Quality Data

In [2]:
epa_path = '../../data/tabular/merged/epa-merged.csv'
df_wq = pd.read_csv(epa_path, low_memory=False)
print(f'epa-wq shape: {df_wq.shape}')

df_wq.head(5)

epa-wq shape: (6581, 57)


,MonitoringLocationIdentifier,ActivityStartDateTime,Ammonia-nitrogen_value,Ammonia-nitrogen_unit,Chloride_value,Chloride_unit,"Chlorophyll a, free of pheophytin_value","Chlorophyll a, free of pheophytin_unit",Copper_value,Copper_unit,...,pH_unit,OrganizationIdentifier,MonitoringLocationName,MonitoringLocationTypeName,HUCEightDigitCode,LatitudeMeasure,LongitudeMeasure,StateCode,CountyCode,ProviderName
0,21IOWA_WQX-10030001,2024-01-02 10:10:00,NaN,NaN,16.0,mg/L,3.0,ug/L,NaN,NaN,...,NaN,21IOWA_WQX,Upper Iowa River near Dorchester,River/Stream,7060002,43.421498,-91.508917,19,5,STORET
1,21IOWA_WQX-10030001,2024-02-01 11:10:00,NaN,NaN,14.0,mg/L,8.0,ug/L,NaN,NaN,...,NaN,21IOWA_WQX,Upper Iowa River near Dorchester,River/Stream,7060002,43.421498,-91.508917,19,5,STORET
2,21IOWA_WQX-10030001,2024-03-12 10:00:00,NaN,NaN,16.0,mg/L,9.0,ug/L,NaN,NaN,...,NaN,21IOWA_WQX,Upper Iowa River near Dorchester,River/Stream,7060002,43.421498,-91.508917,19,5,STORET
3,21IOWA_WQX-10030001,2024-04-08 09:55:00,NaN,NaN,20.0,mg/L,34.0,ug/L,NaN,NaN,...,NaN,21IOWA_WQX,Upper Iowa River near Dorchester,River/Stream,7060002,43.421498,-91.508917,19,5,STORET
4,21IOWA_WQX-10030001,2024-05-07 11:15:00,NaN,NaN,25.0,mg/L,30.0,ug/L,1.6,ug/L,...,NaN,21IOWA_WQX,Upper Iowa River near Dorchester,River/Stream,7060002,43.421498,-91.508917,19,5,STORET


## Step 2: Fetch ISU Climate Station Coordinates from IEM

ISU climate stations are identified by codes (e.g. `IA1533`). We fetch their lat/lon from the IEM GeoJSON endpoint and filter to only stations present in our climate dataset.

In [3]:
climate_path = '../../data/tabular/climate/raw/isu-climate.csv'
df_climate = pd.read_csv(climate_path)
print(f'Climate shape: {df_climate.shape}')
print(f'Unique climate stations: {df_climate["station"].nunique()}')

# Fix date type
df_climate['day'] = pd.to_datetime(df_climate['day'], errors='coerce')
df_climate.dropna(subset=['day'], inplace=True)

climate_stations_in_data = set(df_climate['station'].unique())
print(f'Date range: {df_climate["day"].min().date()} to {df_climate["day"].max().date()}')

Climate shape: (115602, 12)
Unique climate stations: 144
Date range: 2024-01-01 to 2026-03-13


In [4]:
# Fetch station metadata from IEM
IEM_URL = 'https://mesonet.agron.iastate.edu/geojson/network/IACLIMATE.geojson'

response = requests.get(IEM_URL, timeout=30)
response.raise_for_status()
geojson = response.json()

climate_meta_rows = []
for feature in geojson['features']:
    sid = feature['id']
    lon, lat = feature['geometry']['coordinates']
    name = feature['properties'].get('sname', '')
    climate_meta_rows.append({'climate_station': sid, 'climate_station_name': name, 'climate_lat': lat, 'climate_lon': lon})

df_climate_meta = pd.DataFrame(climate_meta_rows)

# Keep only stations present in the climate dataset
df_climate_meta = df_climate_meta[df_climate_meta['climate_station'].isin(climate_stations_in_data)].reset_index(drop=True)

print(f'Climate stations with coordinates: {len(df_climate_meta)}')
df_climate_meta.head(5)

Climate stations with coordinates: 144


,climate_station,climate_station_name,climate_lat,climate_lon
0,IA0000,Iowa Average,41.7500,-93.2500
1,IA0133,ALGONA,43.0700,-94.3000
2,IA0149,Allerton,40.7036,-93.3633
3,IA0157,ALLISON,42.7500,-92.7800
4,IA0197,AMES MUNICIPAL AP,41.9904,-93.6185


## Step 5: Spatial Nearest-Neighbor — Map Each EPA Station to Closest Climate Station

We use `scipy.spatial.cKDTree` with haversine-projected coordinates to find the nearest climate station for each EPA monitoring location.

In [5]:
def haversine_deg_to_rad(lat_lon_deg):
    """Convert (lat, lon) degrees to radians scaled for haversine."""
    return np.radians(lat_lon_deg)

# Build KD-tree from climate station coordinates (in radians for haversine approx)
climate_coords_rad = np.radians(df_climate_meta[['climate_lat', 'climate_lon']].values)
tree = cKDTree(climate_coords_rad)

# Get unique EPA locations with lat/lon
df_epa_locs = df_wq[['MonitoringLocationIdentifier', 'LatitudeMeasure', 'LongitudeMeasure']].dropna().drop_duplicates()
print(f'Unique EPA locations with coordinates: {len(df_epa_locs)}')

epa_coords_rad = np.radians(df_epa_locs[['LatitudeMeasure', 'LongitudeMeasure']].values)

# Query nearest climate station for each EPA location
distances, indices = tree.query(epa_coords_rad, k=1)

# Earth radius ≈ 6371 km; convert radian distance to km
distances_km = distances * 6371

df_epa_locs = df_epa_locs.copy()
df_epa_locs['climate_station'] = df_climate_meta['climate_station'].iloc[indices].values
df_epa_locs['climate_station_name'] = df_climate_meta['climate_station_name'].iloc[indices].values
df_epa_locs['distance_to_climate_station_km'] = distances_km

print(f'\nNearest-neighbor distance stats (km):')
print(df_epa_locs['distance_to_climate_station_km'].describe().round(2))
df_epa_locs.head(5)

Unique EPA locations with coordinates: 612

Nearest-neighbor distance stats (km):
count    612.00
mean      15.23
std        8.49
min        0.17
25%        8.75
50%       14.76
75%       20.39
max       48.16
Name: distance_to_climate_station_km, dtype: float64


,MonitoringLocationIdentifier,LatitudeMeasure,LongitudeMeasure,climate_station,climate_station_name,distance_to_climate_station_km
0,21IOWA_WQX-10030001,43.421498,-91.508917,IA8755,WAUKON,17.149934
16,21IOWA_WQX-10030002,43.111779,-91.264921,IA8755,WAUKON,29.689849
32,21IOWA_WQX-10040002,40.810720,-92.882850,IA6910,RATHBUN DAM,1.916431
48,21IOWA_WQX-10070001,42.573520,-92.506710,IATALO,Waterloo Area,11.912315
64,21IOWA_WQX-10070002,42.315748,-92.193798,IA8568,VINTON,26.963370


## Step 6: Join Climate Station Mapping Back to EPA Data

In [6]:
# Merge climate station assignment into main EPA dataframe
df_epa = df_wq.merge(
    df_epa_locs[['MonitoringLocationIdentifier', 'climate_station', 'climate_station_name', 'distance_to_climate_station_km']],
    on='MonitoringLocationIdentifier',
    how='left'
)

print(f'EPA data with climate station assigned: {df_epa["climate_station"].notna().sum():,} / {len(df_epa):,} rows')
df_epa[['MonitoringLocationIdentifier', 'ActivityStartDateTime', 'climate_station', 'climate_station_name']].head(5)

EPA data with climate station assigned: 6,581 / 6,581 rows


,MonitoringLocationIdentifier,ActivityStartDateTime,climate_station,climate_station_name
0,21IOWA_WQX-10030001,2024-01-02 10:10:00,IA8755,WAUKON
1,21IOWA_WQX-10030001,2024-02-01 11:10:00,IA8755,WAUKON
2,21IOWA_WQX-10030001,2024-03-12 10:00:00,IA8755,WAUKON
3,21IOWA_WQX-10030001,2024-04-08 09:55:00,IA8755,WAUKON
4,21IOWA_WQX-10030001,2024-05-07 11:15:00,IA8755,WAUKON


## Step 7: Merge EPA Data with Climate Data on (climate_station, date)

In [11]:
# Normalize date columns to date-only (no time component) for joining
df_epa['ActivityStartDateTime'] = pd.to_datetime(df_epa['ActivityStartDateTime'], errors='coerce')
df_epa['merge_date'] = df_epa['ActivityStartDateTime'].dt.normalize()
df_climate['merge_date'] = df_climate['day'].dt.normalize()

# Rename climate station column to match
df_climate_merge = df_climate.rename(columns={'station': 'climate_station'})

# Select climate columns to bring in
climate_feature_cols = ['climate_station', 'merge_date', 'doy', 'gdd_40_86', 'high', 'highc', 'low', 'lowc', 'precip', 'snow', 'snowd']
climate_feature_cols = [c for c in climate_feature_cols if c in df_climate_merge.columns]
df_climate_features = df_climate_merge[climate_feature_cols].drop_duplicates(subset=['climate_station', 'merge_date'])

# Merge on (climate_station, merge_date)
df_merged = df_epa.merge(
    df_climate_features,
    on=['climate_station', 'merge_date'],
    how='left'
)

# Drop helper column
df_merged.drop(columns=['merge_date'], inplace=True)

print(f'Final merged shape: {df_merged.shape}')
climate_match_rate = df_merged['high'].notna().sum() / len(df_merged) * 100
print(f'Climate match rate: {climate_match_rate:.1f}%')
df_merged.head(3)

Final merged shape: (6581, 69)
Climate match rate: 99.0%


,MonitoringLocationIdentifier,ActivityStartDateTime,Ammonia-nitrogen_value,Ammonia-nitrogen_unit,Chloride_value,Chloride_unit,"Chlorophyll a, free of pheophytin_value","Chlorophyll a, free of pheophytin_unit",Copper_value,Copper_unit,...,distance_to_climate_station_km,doy,gdd_40_86,high,highc,low,lowc,precip,snow,snowd
0,21IOWA_WQX-10030001,2024-01-02 10:10:00,NaN,NaN,16.0,mg/L,3.0,ug/L,NaN,NaN,...,17.149934,2,0.0,23.0,-5.0,18.0,-7.8,0.0,0.0,0.0
1,21IOWA_WQX-10030001,2024-02-01 11:10:00,NaN,NaN,14.0,mg/L,8.0,ug/L,NaN,NaN,...,17.149934,32,2.5,45.0,7.2,28.0,-2.2,0.0,0.0,0.0
2,21IOWA_WQX-10030001,2024-03-12 10:00:00,NaN,NaN,16.0,mg/L,9.0,ug/L,NaN,NaN,...,17.149934,72,13.5,67.0,19.4,39.0,3.9,0.0,0.0,0.0


## Step 8: Summary and Save

In [13]:
print('=== Merged Dataset Summary ===')
print(f'Total rows:              {len(df_merged):,}')
print(f'Unique EPA locations:    {df_merged["MonitoringLocationIdentifier"].nunique():,}')
print(f'Unique climate stations: {df_merged["climate_station"].nunique():,}')
print(f'Date range:              {df_merged["ActivityStartDateTime"].min().date()} to {df_merged["ActivityStartDateTime"].max().date()}')
print(f'Climate match rate:      {climate_match_rate:.1f}%')
print(f'\nColumns: {df_merged.columns.tolist()}')
print(f'\nMissing values:')
print(df_merged.isnull().sum()[df_merged.isnull().sum() > 0])

=== Merged Dataset Summary ===
Total rows:              6,581
Unique EPA locations:    612
Unique climate stations: 109
Date range:              2024-01-02 to 2025-12-25
Climate match rate:      99.0%

Columns: ['MonitoringLocationIdentifier', 'ActivityStartDateTime', 'Ammonia-nitrogen_value', 'Ammonia-nitrogen_unit', 'Chloride_value', 'Chloride_unit', 'Chlorophyll a, free of pheophytin_value', 'Chlorophyll a, free of pheophytin_unit', 'Copper_value', 'Copper_unit', 'Dissolved oxygen (DO)_value', 'Dissolved oxygen (DO)_unit', 'Escherichia coli_value', 'Escherichia coli_unit', 'Kjeldahl nitrogen_value', 'Kjeldahl nitrogen_unit', 'Microcystin_value', 'Microcystin_unit', 'Nitrate_value', 'Nitrate_unit', 'Nitrate + Nitrite_value', 'Nitrate + Nitrite_unit', 'Nitrite_value', 'Nitrite_unit', 'Orthophosphate_value', 'Orthophosphate_unit', 'Precipitation 24hr prior to monitoring event amount_value', 'Precipitation 24hr prior to monitoring event amount_unit', 'Specific conductance_value', 'Speci

In [14]:
out_path = '../../data/tabular/merged'
os.makedirs(out_path, exist_ok=True)
out_file = f'{out_path}/epa-climate-merged.csv'
df_merged.to_csv(out_file, index=False)
print(f'Saved {len(df_merged):,} rows → {out_file}')

Saved 6,581 rows → ../../data/tabular/merged/epa-climate-merged.csv


In [15]:
df_merged.head(10)

,MonitoringLocationIdentifier,ActivityStartDateTime,Ammonia-nitrogen_value,Ammonia-nitrogen_unit,Chloride_value,Chloride_unit,"Chlorophyll a, free of pheophytin_value","Chlorophyll a, free of pheophytin_unit",Copper_value,Copper_unit,...,distance_to_climate_station_km,doy,gdd_40_86,high,highc,low,lowc,precip,snow,snowd
0,21IOWA_WQX-10030001,2024-01-02 10:10:00,NaN,NaN,16.0,mg/L,3.0,ug/L,NaN,NaN,...,17.149934,2,0.0,23.0,-5.0,18.0,-7.8,0.0000,0.0,0.0
1,21IOWA_WQX-10030001,2024-02-01 11:10:00,NaN,NaN,14.0,mg/L,8.0,ug/L,NaN,NaN,...,17.149934,32,2.5,45.0,7.2,28.0,-2.2,0.0000,0.0,0.0
2,21IOWA_WQX-10030001,2024-03-12 10:00:00,NaN,NaN,16.0,mg/L,9.0,ug/L,NaN,NaN,...,17.149934,72,13.5,67.0,19.4,39.0,3.9,0.0000,0.0,0.0
3,21IOWA_WQX-10030001,2024-04-08 09:55:00,NaN,NaN,20.0,mg/L,34.0,ug/L,NaN,NaN,...,17.149934,99,2.5,45.0,7.2,39.0,3.9,0.2500,0.0,0.0
4,21IOWA_WQX-10030001,2024-05-07 11:15:00,NaN,NaN,25.0,mg/L,30.0,ug/L,1.6,ug/L,...,17.149934,128,21.5,70.0,21.1,53.0,11.7,0.7400,0.0,0.0
5,21IOWA_WQX-10030001,2024-06-12 10:00:00,NaN,NaN,21.0,mg/L,14.0,ug/L,1.3,ug/L,...,17.149934,164,27.0,79.0,26.1,55.0,12.8,0.0000,0.0,0.0
6,21IOWA_WQX-10030001,2024-07-10 09:10:00,NaN,NaN,17.0,mg/L,3.0,ug/L,2.0,ug/L,...,17.149934,192,31.0,78.0,25.6,64.0,17.8,0.0000,0.0,0.0
7,21IOWA_WQX-10030001,2024-08-08 10:35:00,NaN,NaN,14.0,mg/L,46.0,ug/L,1.7,ug/L,...,17.149934,221,30.0,76.0,24.4,64.0,17.8,0.0000,0.0,0.0
8,21IOWA_WQX-10030001,2024-09-10 10:00:00,0.07,mg/L,15.0,mg/L,5.0,ug/L,2.0,ug/L,...,17.149934,254,25.5,78.0,25.6,53.0,11.7,0.0001,0.0,0.0
9,21IOWA_WQX-10030001,2024-10-02 10:45:00,NaN,NaN,16.0,mg/L,7.0,ug/L,NaN,NaN,...,17.149934,276,13.0,63.0,17.2,43.0,6.1,0.0000,0.0,0.0


## (Optional) Station-to-Station Mapping Reference

In [16]:
# Save the EPA → climate station mapping for reference
df_epa_locs[['MonitoringLocationIdentifier', 'LatitudeMeasure', 'LongitudeMeasure',
              'climate_station', 'climate_station_name', 'distance_to_climate_station_km']].to_csv(
    f'{out_path}/epa-to-climate-station-map.csv', index=False
)
print(f'Saved station mapping → {out_path}/epa-to-climate-station-map.csv')
df_epa_locs.sort_values('distance_to_climate_station_km').head(10)

Saved station mapping → ../../data/tabular/merged/epa-to-climate-station-map.csv


,MonitoringLocationIdentifier,LatitudeMeasure,LongitudeMeasure,climate_station,climate_station_name,distance_to_climate_station_km
2014,21IOWA_WQX-22040001,40.824972,-92.894015,IA6910,RATHBUN DAM,0.168489
435,21IOWA_WQX-10490004,42.070737,-90.695404,IA5131,MAQUOKETA-2-W,0.517581
2722,EPA_R7_WQX-01-NMQ-90,42.299388,-91.012551,IA1257,CASCADE,0.831082
2482,21IOWA_WQX-22890004,40.711232,-91.970015,IA4389,KEOSAUQUA,0.974959
2108,21IOWA_WQX-22240002,42.022643,-95.324804,IA2171,DENISON,1.001519
1913,21IOWA_WQX-21890001,40.709206,-91.971532,IA4389,KEOSAUQUA,1.212267
1994,21IOWA_WQX-22010001,41.298198,-94.481333,IA3438,GREENFIELD,1.276003
2427,21IOWA_WQX-22790004,41.731706,-92.732963,IA3473,GRINNELL-3-SW,1.342698
862,21IOWA_WQX-10890001,40.728232,-91.960615,IA4389,KEOSAUQUA,1.388130
2503,21IOWA_WQX-22940001,42.586293,-94.192441,IAD002,Iowa Drought Region 2 - Northcentral,1.460330
